# [0] Naive Trial with TTT Layer + KcELECTRA

:reference: https://github.com/test-time-training/ttt-lm-pytorch

:suggesting paper: https://arxiv.org/abs/2407.04620

## Imports

In [ ]:
import env

In [ ]:
from epidec.datasets import BalancedSWUnivDaconDataset

from transformers import ElectraModel, AutoTokenizer, BitsAndBytesConfig
from torch.utils.data import DataLoader
from torch import nn
import torch

from lattent import TTTPreTrainedModel, TTTConfig, Block as TTTBlock, RMSNorm as TTTRMSNorm
from sklearn.metrics import roc_auc_score

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import sys
import gc
import re

In [ ]:
for test in tqdm(range(1000), desc="Testing tqdm"):
    pass

### Check GPU Availability

In [ ]:
!nvidia-smi

In [ ]:
# Set CUDA Device
device_num = 0

if torch.cuda.is_available() and device_num != -1:
    torch.cuda.set_device(device_num)
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    device_num = -1  # cpu
print(f"INFO: Using device - {device}:{device_num}")

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = BalancedSWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

## Define Model

In [ ]:
base_model_id = "beomi/KcELECTRA-base-v2022"

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
from typing import Optional

class TTTForNaiveTextDetection(TTTPreTrainedModel):
    def __init__(
        self,
        config: TTTConfig = TTTConfig(num_hidden_layers=2),
        quantization_config: Optional[BitsAndBytesConfig] = None
    ):
        base = ElectraModel.from_pretrained(
            base_model_id,
            trust_remote_code=True,
            quantization_config=quantization_config
        )
        self.padding_idx = base.config.pad_token_id
        self.vocab_size = base.config.vocab_size

        config.vocab_size = self.vocab_size  # Electra vocab size
        config.hidden_size = base.config.hidden_size  # Electra hidden size
        config.intermediate_size = config.hidden_size // 2
        super().__init__(config)

        # 0. Register base model
        self.base = base
        self.base.encoder.layer = base.encoder.layer[:-config.num_hidden_layers]  # Remove the last n layers
        for param in self.base.parameters():
            param.requires_grad = False  # Freeze the base model parameters

        # 2. Main TTT model
        self.model = nn.ModuleList([TTTBlock(config, layer_idx) for layer_idx in range(config.num_hidden_layers)])
        self.norm = TTTRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.gradient_checkpointing = False

        # 3. Final classification layer
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(config.hidden_size, 1),
        )

        # 4. Initialize weights and apply final processing
        self.post_init()

    def forward(
        self,
        input_ids: list[torch.LongTensor],
        attention_mask: list[torch.Tensor]
    ) -> torch.Tensor:
        hidden_states = []
        with torch.no_grad():
            for inputs, masks in zip(input_ids, attention_mask):
                processed = self.base(input_ids=inputs, attention_mask=masks).last_hidden_state
                hidden_states.append(processed[:, 1:-2, :])

        hidden_states = torch.cat(hidden_states, dim=1)

        seqlen_offset = 0
        position_ids = torch.arange(
            seqlen_offset, seqlen_offset + hidden_states.shape[1],
            dtype=torch.long, device=hidden_states.device,
        ).unsqueeze(0)

        for decoder_layer in self.model:
            hidden_states = decoder_layer(
                hidden_states, position_ids=position_ids
            )

        hidden_states = self.norm(hidden_states[:, -1, :])
        return self.classifier(hidden_states)

In [ ]:
model = TTTForNaiveTextDetection()
model.to(device)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

def tokenize(batch):
    return tokenizer(
        batch, return_tensors="pt", return_token_type_ids=False,
        padding="longest" if len(batch) > 1 else False
    ).to(device)

In [ ]:
tokenize("Hello, my dog is cute")

## Train and Evaluate

In [ ]:
eval_steps = 200

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(test_dataset)//3)

In [ ]:
BATCH_SIZE = 1, 1, 1

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE[0], shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE[1], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE[2], shuffle=False, collate_fn=lambda x: x)

In [ ]:
def to_label(scores, threshold=0.5):
    return [1 if score > threshold else 0 for score in scores]

def accuracy(scores, preds, labels):
    correct, true_human, false_human = 0, 0, 0
    for score, pred, label in zip(scores, preds, labels):
        if pred == label:
            if label == 0: true_human += 1
            correct += 1
            print(f"INFO: Correct prediction - expected {label}, got {pred} [{score.tolist()}]")
        else:
            if label == 0: false_human += 1
            print(f"ERROR: Incorrect prediction - expected {label}, got {pred} [{score.tolist()}]")
    return correct, true_human, false_human

In [ ]:
def valid_sample(sample_amount=5):
    while True:
        human_count, ai_count = 0, 0
        for texts, labels in DataLoader(valid_dataset, batch_size=1, shuffle=True):
            if labels[0] == 0:
                if human_count >= sample_amount: continue
                human_count += 1
            else:
                if ai_count >= sample_amount: continue
                ai_count += 1

            yield texts, labels
            if human_count >= sample_amount and ai_count >= sample_amount:
                return

In [ ]:
def split_sentence(text):
    sentences = re.split(r'(?<=[.?!])\s*', text)
    return [s for s in sentences if s]

In [ ]:
valid_amount = 100

with (
    tqdm(train_loader, desc="[Training]") as progress,
    tqdm(range(valid_amount*2), desc="[Validating]") as v_progress
):
    train_loss, train_preds, train_labels = [], [], []
    for step, (texts, labels) in enumerate(progress):
        model.train()
        try:
            optimizer.zero_grad()
            input_ids, attention_masks = [], []
            for tokenized in [tokenize(t) for t in split_sentence(texts[0])]:
                input_ids.append(tokenized['input_ids'].to(device))
                attention_masks.append(tokenized['attention_mask'].to(device))

            logits = model(input_ids=input_ids, attention_mask=attention_masks)
            loss = criterion(logits, labels.unsqueeze(0).float().to(device)) * (1.4 if labels[0].item() == 1 else 1)
            train_preds.append(torch.sigmoid(logits)[0][0].item())
            train_labels.append(labels[0].item())
            train_loss.append(loss.item())
            loss.backward()

            optimizer.step()
            scheduler.step()
            progress.set_description(f"[Training] Step: {step+1}, Loss: {sum(train_loss)/len(train_loss):.6f}, ROC AUC: {roc_auc_score(train_labels, train_preds):.6f}")
        except Exception as e:
            print(e, file=sys.stderr)

        if (step+1) % eval_steps == 0:
            model.eval()
            train_loss, train_preds, train_labels, valid_preds, valid_labels = [], [], [], [], []
            corrects, errors, true_human, false_human = 0, 0, 0, 0
            v_progress.reset()
            for texts, labels in valid_sample(valid_amount):
                try:
                    with torch.no_grad():
                        input_ids, attention_masks = [], []
                        for tokenized in [tokenize(t) for t in split_sentence(texts[0])]:
                            input_ids.append(tokenized['input_ids'].to(device))
                            attention_masks.append(tokenized['attention_mask'].to(device))

                        logits = model(input_ids=input_ids, attention_mask=attention_masks)
                        scores = torch.sigmoid(logits)[0]
                        preds = to_label(scores)

                        valid_preds.append(scores[0].item())
                        valid_labels.append(labels[0].item())
                        c, th, fh = accuracy(scores, preds, labels.tolist())
                        corrects += c
                        errors += len(labels) - c
                        true_human += th
                        false_human += fh
                except Exception as e:
                    print(e, file=sys.stderr)

                v_progress.set_description(f"[Validating] Correct: {corrects/(corrects+errors):.6%} [H: {true_human}, A: {corrects-true_human}], Errors: {errors} [H: {false_human}, A: {errors-false_human}], ROC AUC: {roc_auc_score(valid_labels, valid_preds):.6f}")
                v_progress.update(1)

In [ ]:
with tqdm(valid_loader, desc="[Validating]") as progress:
    corrects, errors, true_human, false_human, valid_preds, valid_labels = 0, 0, 0, 0, [], []
    model.eval()
    for texts, labels in progress:
        torch.cuda.empty_cache()
        gc.collect()

        try:
            with torch.no_grad():
                input_ids, attention_masks = [], []
                for tokenized in [tokenize(t) for t in split_sentence(texts[0])]:
                    input_ids.append(tokenized['input_ids'].to(device))
                    attention_masks.append(tokenized['attention_mask'].to(device))

                logits = model(input_ids=input_ids, attention_mask=attention_masks)
                scores = torch.sigmoid(logits)[0]
                preds = to_label(scores)

                valid_preds.append(scores[0].item())
                valid_labels.append(labels[0].item())
                c, th, fh = accuracy(scores, preds, labels.tolist())
                corrects += c
                errors += len(labels) - c
                true_human += th
                false_human += fh
        except Exception as e:
            if "CUDA" in str(e):
                print(f"ERROR: {e} - {texts}")
            else:
                raise e

        progress.set_description(f"[Validating] Correct: {corrects/(corrects+errors):.6%} [H: {true_human}, A: {corrects-true_human}], Errors: {errors} [H: {false_human}, A: {errors-false_human}], ROC AUC: {roc_auc_score(valid_labels, valid_preds):.6f}")

In [ ]:
results = []
with tqdm(test_loader, desc="[Testing]") as progress:
    humans, ais = 0, 0
    model.eval()
    for datas in progress:
        texts, _ = zip(*datas)
        torch.cuda.empty_cache()
        gc.collect()

        try:
            with torch.no_grad():
                input_ids, attention_masks = [], []
                for tokenized in [tokenize(t) for t in split_sentence(texts[0])]:
                    input_ids.append(tokenized['input_ids'].to(device))
                    attention_masks.append(tokenized['attention_mask'].to(device))

                logits = model(input_ids=input_ids, attention_mask=attention_masks)
                scores = torch.sigmoid(logits)[0]
                results.append(scores.cpu()[0].item())
                preds = to_label(scores)[0]
                if preds == 0:
                    humans += 1
                else:
                    ais += 1
        except Exception as e:
            if "CUDA" in str(e):
                print(f"ERROR: {e} - {texts}")
            else:
                raise e

        progress.set_description(f"[Testing] Human: {humans/len(test_dataset):.2%}, Ai: {ais/len(test_dataset):.2%}")

In [ ]:
sub = pd.read_csv("./data/swuniv_dacon/" + test_dataset.submission_file, encoding='utf-8-sig')
sub

In [ ]:
sub['generated'] = results
sub

In [ ]:
plt.figure(figsize=(12, 7))
sns.histplot(data=sub, x='generated', kde=True, bins=50)
plt.title("Prediction Probability Distribution", fontsize=16)
plt.xlabel("Predicted Probability (Generated = 1)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)

plt.show()
print(sub['generated'].describe())

In [ ]:
sub.to_csv("./data/submission_naive_ttt.csv", index=False, encoding='utf-8-sig')